# Solutions — try/catch/finally → Python Exception Idioms

Exercise notebook: `04_java_to_python/05_exceptions.ipynb`. Each solution has: working code, an explanation, complexity, common mistakes, and a Java-style vs Pythonic comparison where it teaches something.

## Exercise 1 — 🟢 Easy · try / except / else / finally

**Working solution**

In [ ]:
import json
from pathlib import Path

def load_json_config(path):
    path = Path(path)
    try:
        text = path.read_text(encoding="utf-8")
    except FileNotFoundError:
        return {}
    try:
        data = json.loads(text)
    except json.JSONDecodeError as exc:
        raise ValueError(f"invalid JSON in {path.name}: {exc.msg}") from exc
    else:
        if not isinstance(data, dict):
            raise ValueError(f"{path.name} must contain a JSON object")
        return data

**Explanation**

Keep each `try` block tiny so you only catch what you expect from that one operation. `else` holds the code that should run only on
success — and whose exceptions you *don't* want caught by the handler above.

**Complexity:** O(file size).

**Common mistakes**

- One giant `try` around everything with `except Exception`.
- Catching `FileNotFoundError` *and* JSON errors together and returning `{}` for both.

## Exercise 2 — 🟡 Medium · Fallback on transient errors only

**Working solution**

In [ ]:
def fetch_with_fallback(primary, fallback):
    try:
        return primary(), "primary"
    except (ConnectionError, TimeoutError):
        return fallback(), "fallback"

**Explanation**

Catch only what you can handle. An exception raised inside an `except` block automatically records the original as `__context__`
("During handling of the above exception, another exception occurred") — Java needs `addSuppressed` for that.

**Complexity:** O(1).

**Common mistakes**

- `except Exception:` → hides bugs as 'used fallback'.
- Returning `None` on failure and making callers guess.

## Exercise 3 — 🟡 Medium · Report all failures with ExceptionGroup

**Working solution**

In [ ]:
def validate_or_raise(records, validator):
    errors = []
    for index, record in enumerate(records):
        try:
            validator(record)
        except ValueError as exc:
            exc.add_note(f"record index {index}")
            errors.append(exc)
    if errors:
        raise ExceptionGroup(f"{len(errors)} invalid records", errors)
    return records

**Explanation**

`ExceptionGroup` carries several independent failures at once; `except*` lets callers handle each type separately. `add_note` attaches
context shown in tracebacks — no need for a custom exception just to add 'which record'.

**Complexity:** O(n).

**Common mistakes**

- Raising only the first error and forcing users to fix-and-rerun one by one.

## Exercise 4 — 🟠 Hard · Translate low-level errors at a boundary

**Working solution**

In [ ]:
import json
from pathlib import Path

class StorageError(Exception):
    def __init__(self, message, path):
        super().__init__(message)
        self.path = str(path)

def load_records(path):
    path = Path(path)
    try:
        text = path.read_bytes().decode("utf-8")
    except FileNotFoundError as exc:
        raise StorageError(f"not found: {path}", path) from exc
    except UnicodeDecodeError as exc:
        raise StorageError(f"encoding error: {path}", path) from exc
    except OSError as exc:
        raise StorageError(f"cannot read: {path}", path) from exc
    try:
        data = json.loads(text)
    except json.JSONDecodeError as exc:
        raise StorageError(f"corrupt: {path}", path) from exc
    if not isinstance(data, list) or not all(isinstance(r, dict) for r in data):
        raise StorageError(f"corrupt: {path}", path)
    return data

**Explanation**

Exception order matters: `FileNotFoundError` is a subclass of `OSError`, so catch it first. Translating at module boundaries keeps callers
coupled to *your* error type, while `from exc` preserves the low-level detail for debugging.

**Complexity:** O(file size).

**Common mistakes**

- `except OSError` before `except FileNotFoundError` → the specific branch never runs.
- Translating without `from` and losing the root cause.

## Challenge — 🔴 Very Hard · Atomic file writes with guaranteed cleanup

**Working solution**

In [ ]:
import json
import os
import tempfile
from pathlib import Path

def write_json_atomic(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, tmp_name = tempfile.mkstemp(dir=path.parent, prefix=f".{path.name}.", suffix=".tmp")
    try:
        with os.fdopen(fd, "w", encoding="utf-8") as f:
            json.dump(payload, f, indent=2)
            f.flush()
            os.fsync(f.fileno())
        os.replace(tmp_name, path)
    except BaseException:
        Path(tmp_name).unlink(missing_ok=True)
        raise

**Explanation**

Write-then-rename is the standard trick for atomic publication (S3 has no rename, so there you write to a staging prefix and copy/commit
a manifest instead). A bare `raise` re-raises the in-flight exception with its traceback intact. Catching `BaseException` here is correct because we
always re-raise — cleanup must happen even on Ctrl+C.

**Complexity:** O(payload size).

**Common mistakes**

- Temp file in `/tmp` on another filesystem — `os.replace` across devices fails.
- `raise e` instead of `raise` (fine in Py3, but bare `raise` is the idiom).
- Swallowing the exception after cleanup.